In [1]:
import json
import os
import subprocess
import sys
from pathlib import Path
from tempfile import TemporaryDirectory

import numpy as np

# Add the project root to the Python path so we can import from it
sys.path.insert(0, str(Path(".").resolve().parent))

os.environ["POLARS_OOC_MEMORY_BUDGET_MB"] = "100"  # disable Polars' own memory budget
os.environ["POLARS_ENGINE_AFFINITY"] = "streaming"
os.environ["POLARS_STREAMING_CHUNK_SIZE"] = "2000"  # small chunks to stress memory
os.environ["POLARS_VERBOSE"] = "0"  # enable verbose logging to stdout for debugging
os.environ["POLARS_MAX_THREADS"] = "1"  # limit to one thread to keep memory usage more predictable

import polars as pl
from data_warehousing_with_polars.incremental import _DeltaCdfSource, incremental
from deltalake import write_deltalake

from utils.sample_data import write_partitioned_measurements

_IMPL = Path(".").parent / "_memory_analysis_impl.py"
assert _IMPL.exists(), f"Expected {_IMPL} to exist"


def cdf_seed(n_partitions: int, rows_per_partition: int) -> pl.DataFrame:
    return pl.concat([
        pl.DataFrame({
            "measurement": [f"measurement_{i}"] * rows_per_partition,
            "value": np.random.rand(rows_per_partition),
        })
        for i in range(n_partitions)
    ])


def _run_subprocess(name: str, tmp_path: Path, n_partitions: int, rows_per_partition: int) -> str:
    """Spawn a fresh interpreter to run ``run_<name>`` in ``_memory_analysis_impl.py``."""
    result = subprocess.run(
        [
            sys.executable,
            str(_IMPL),
            name,
            str(tmp_path),
            str(n_partitions),
            str(rows_per_partition),
        ],
        capture_output=True,
        text=True,
        env={**os.environ},
    )
    output = (result.stdout + result.stderr).strip()
    if result.returncode != 0:
        raise RuntimeError(f"Memory worker '{name}' failed:\n{output}")
    return output


def _measure(name: str, tmp_path: Path, n_partitions: int, rows_per_partition: int) -> dict:
    """Run the measured worker subprocess and parse its ``{"peak_rss_mb": ...}`` output.

    All dataset preparation happens in the notebook (this process) before this
    is called — see each ``prepare_*`` function below — so every worker
    subprocess only ever does the one operation being measured.
    """
    output = _run_subprocess(name, tmp_path, n_partitions, rows_per_partition)
    print(f"Output from memory worker '{name}':\n{output}")
    return json.loads(output.splitlines()[-1])


rows_per_partition = 100_000
num_partitions = 200


In [2]:
def prepare_src(tmp_path: Path, n_partitions: int, rows_per_partition: int) -> float:
    """Writes the harness's first batch — the common case every worker starts from."""
    src = tmp_path / "src"
    src.mkdir(parents=True, exist_ok=True)
    return write_partitioned_measurements(src, n_partitions, rows_per_partition)


def prepare_by_partition_cdf(tmp_path: Path, n_partitions: int, rows_per_partition: int) -> float:
    """Creates cdf_source, runs a throwaway pipeline once to establish the
    watermark/target, and appends the new CDF-visible commit the measured run
    picks up as its single-use batch. Returns that new batch's size (MB).
    """
    source = tmp_path / "cdf_source"
    write_deltalake(
        str(source),
        cdf_seed(n_partitions, rows_per_partition).to_arrow(),
        mode="overwrite",
        partition_by=["measurement"],
        configuration={"delta.enableChangeDataFeed": "true"},
    )

    @incremental(
        source=str(source),
        target=str(tmp_path / "target"),
        file_format="delta",
        merge_on=None,
        partition_by="measurement",
        by_partition=True,
        by_partition_workers=1,
    )
    def pipeline(lf: pl.LazyFrame) -> pl.LazyFrame:
        return lf

    pipeline.run()  # from_version=None, establishes the watermark

    new_batch = cdf_seed(n_partitions, rows_per_partition)
    write_deltalake(str(source), new_batch.to_arrow(), mode="append", partition_by=["measurement"])
    return new_batch.estimated_size(unit="mb")


def prepare_upsert(tmp_path: Path, n_partitions: int, rows_per_partition: int) -> float:
    """Creates the table (a first run against src/), then writes a second file
    batch — the new data the measured run's ``_upsert_overwrite`` call processes.
    """
    prepare_src(tmp_path, n_partitions, rows_per_partition)

    @incremental(
        source=str(tmp_path / "src"),
        target=str(tmp_path / "target"),
        merge_on="channel",
        partition_by="measurement",
        by_partition=True,
        by_partition_workers=1,
    )
    def pipeline(lf: pl.LazyFrame) -> pl.LazyFrame:
        return lf

    pipeline.run()
    return write_partitioned_measurements(
        tmp_path / "src", n_partitions, rows_per_partition, suffix="00002"
    )


def prepare_fan_in_cdf_and_file(
    tmp_path: Path, n_partitions: int, rows_per_partition: int
) -> float:
    """Creates both sources, runs the pipeline once to establish both cursors,
    then writes new data to both — what the measured run processes.
    """
    prepare_src(tmp_path, n_partitions, rows_per_partition)
    cdf_source = tmp_path / "cdf_source"
    write_deltalake(
        str(cdf_source),
        cdf_seed(n_partitions, rows_per_partition).to_arrow(),
        mode="overwrite",
        partition_by=["measurement"],
        configuration={"delta.enableChangeDataFeed": "true"},
    )

    @incremental(
        source=[str(tmp_path / "src"), _DeltaCdfSource(str(cdf_source))],
        target=str(tmp_path / "target"),
        merge_on=None,
        partition_by="measurement",
        by_partition=True,
        by_partition_workers=1,
    )
    def pipeline(lf_file: pl.LazyFrame, lf_cdf: pl.LazyFrame) -> pl.LazyFrame:
        return pl.concat([lf_file, lf_cdf], how="diagonal_relaxed")

    pipeline.run()

    file_size = write_partitioned_measurements(
        tmp_path / "src", n_partitions, rows_per_partition, suffix="00002"
    )
    new_cdf_batch = cdf_seed(n_partitions, rows_per_partition)
    write_deltalake(
        str(cdf_source), new_cdf_batch.to_arrow(), mode="append", partition_by=["measurement"]
    )
    return file_size + new_cdf_batch.estimated_size(unit="mb")


def prepare_compact_every(
    tmp_path: Path, n_partitions: int, rows_per_partition: int, rounds: int = 3
) -> float:
    """Accumulates ``rounds`` small, uncompacted prior commits (simulating real
    file fragmentation from repeated small batches) before the measured run's
    ``compact_every=1`` triggers ``maintain()`` — otherwise there'd be nothing
    to compact on a freshly-created, already-tidy table.
    """
    prepare_src(tmp_path, n_partitions, rows_per_partition)

    @incremental(
        source=str(tmp_path / "src"),
        target=str(tmp_path / "target"),
        merge_on=None,
        partition_by="measurement",
        by_partition=True,
        by_partition_workers=1,
    )
    def pipeline(lf: pl.LazyFrame) -> pl.LazyFrame:
        return lf

    pipeline.run()  # first batch

    for r in range(rounds):
        write_partitioned_measurements(
            tmp_path / "src", n_partitions, rows_per_partition, suffix=f"prior{r:02d}"
        )
        pipeline.run()  # accumulates fragmentation; compact_every unset here on purpose

    # One more new batch for the measured subprocess's compact_every=1 run to process.
    return write_partitioned_measurements(
        tmp_path / "src", n_partitions, rows_per_partition, suffix="final"
    )


PREPARE = {
    "by_partition_cdf": prepare_by_partition_cdf,
    "upsert": prepare_upsert,
    "fan_in_cdf_and_file": prepare_fan_in_cdf_and_file,
    "compact_every": prepare_compact_every,
}


# Analyze Memory Usage of Different Incremental Processing Strategies

In [ ]:
workers = [
    "pure_polars",
    "all",
    "by_partition",
    "scd2",
    "scd4",
    "by_partition_cdf",
    "upsert",
    "fan_in_cdf_and_file",
    "compact_every",
]

results = []

for i in range(1, num_partitions + 1, 20):
    for worker in workers:
        with TemporaryDirectory() as tmp_dir:
            tmp_path = Path(tmp_dir)
            prepare = PREPARE.get(worker, prepare_src)
            dataset_size_mb = prepare(tmp_path, i, rows_per_partition)

            result = _measure(worker, tmp_path, i, rows_per_partition)
            print(
                f"[{worker}] n_partitions={i}: "
                f"Peak RSS: {result['peak_rss_mb']:.1f} MB, "
                f"Dataset size: {dataset_size_mb:.1f} MB"
            )
            result.update({
                "dataset_size_mb": dataset_size_mb,
                "worker": worker,
                "n_partitions": i,
                "rows_per_partition": rows_per_partition,
            })
            results.append(result)

results_df = pl.DataFrame(results)


Output from memory worker 'pure_polars':
{"peak_rss_mb": 50.96875}
[pure_polars] n_partitions=1: Peak RSS: 51.0 MB, Dataset size: 2.3 MB
Output from memory worker 'pure_polars':
{"peak_rss_mb": 198.890625}
[pure_polars] n_partitions=21: Peak RSS: 198.9 MB, Dataset size: 49.1 MB
Output from memory worker 'pure_polars':
{"peak_rss_mb": 268.421875}
[pure_polars] n_partitions=41: Peak RSS: 268.4 MB, Dataset size: 96.8 MB
Output from memory worker 'pure_polars':
{"peak_rss_mb": 344.390625}
[pure_polars] n_partitions=61: Peak RSS: 344.4 MB, Dataset size: 144.5 MB
Output from memory worker 'pure_polars':
{"peak_rss_mb": 399.546875}
[pure_polars] n_partitions=81: Peak RSS: 399.5 MB, Dataset size: 192.2 MB
Output from memory worker 'pure_polars':
{"peak_rss_mb": 453.75}
[pure_polars] n_partitions=101: Peak RSS: 453.8 MB, Dataset size: 239.9 MB
Output from memory worker 'pure_polars':
{"peak_rss_mb": 518.765625}
[pure_polars] n_partitions=121: Peak RSS: 518.8 MB, Dataset size: 289.5 MB
Output fr

KeyboardInterrupt: 

In [9]:
import altair as alt

# Each incremental code path gets its own identifiable color (Tableau-10-style,
# avoiding blue/black — those are reserved below); polars stays a fixed
# reference blue; dataset size is its own thin dashed black reference line.
_worker_domain = [
    "pure_polars",
    "all",
    "by_partition",
    "scd2",
    "scd4",
    "by_partition_cdf",
    "upsert",
    "fan_in_cdf_and_file",
    "compact_every",
    "dataset_size",
]
_color_range = [
    "#0075ff",  # pure_polars — polars blue
    "#F28E2B",  # all
    "#59A14F",  # by_partition
    "#E15759",  # scd2
    "#EDC948",  # scd4
    "#B07AA1",  # by_partition_cdf
    "#FF9DA7",  # upsert
    "#9C755F",  # fan_in_cdf_and_file
    "#BAB0AC",  # compact_every
    "#000000",  # dataset_size
]
# Every code path drawn the same weight — no bold treatment; only dataset
# size is deliberately thinner, since it's a reference line, not a result.
_width_range = [2, 2, 2, 2, 2, 2, 2, 2, 2, 1]
_dash_range = [[1, 0]] * 9 + [[6, 4]]  # solid for every worker, dashed for dataset size

long_rows = []
for r in results_df.iter_rows(named=True):
    long_rows.append({
        "n_partitions": r["n_partitions"],
        "worker": r["worker"],
        "series": r["worker"],
        "value": r["peak_rss_mb"],
    })
    long_rows.append({
        "n_partitions": r["n_partitions"],
        "worker": r["worker"],
        "series": "dataset_size",
        "value": r["dataset_size_mb"],
    })

long_df = pl.DataFrame(long_rows)

alt.Chart(long_df).mark_line().encode(
    x=alt.X("n_partitions:Q", title="n_partitions"),
    y=alt.Y("value:Q", title="peak RSS / dataset size (MB)"),
    color=alt.Color(
        "series:N",
        scale=alt.Scale(domain=_worker_domain, range=_color_range),
        legend=alt.Legend(title=None),
    ),
    strokeWidth=alt.StrokeWidth(
        "series:N", scale=alt.Scale(domain=_worker_domain, range=_width_range), legend=None
    ),
    strokeDash=alt.StrokeDash(
        "series:N", scale=alt.Scale(domain=_worker_domain, range=_dash_range), legend=None
    ),
    detail="worker:N",
    tooltip=["worker", "n_partitions", "value", "series"],
).properties(width=700, height=420)


alt.Chart(...)

In [5]:
results_df.sort("n_partitions", "worker")


peak_rss_mb,dataset_size_mb,worker,n_partitions,rows_per_partition
f64,f64,str,i64,i64
80.75,2.288818,"""all""",1,100000
79.90625,2.288818,"""by_partition""",1,100000
110.015625,2.002716,"""by_partition_cdf""",1,100000
187.75,2.288818,"""compact_every""",1,100000
144.265625,4.291534,"""fan_in_cdf_and_file""",1,100000
…,…,…,…,…
439.734375,824.832916,"""fan_in_cdf_and_file""",181,100000
675.484375,438.308716,"""pure_polars""",181,100000
340.421875,438.308716,"""scd2""",181,100000


In [ ]:
import altair as alt

subset = results_df.filter(pl.col("worker").is_in(["pure_polars", "by_partition"]))

long_df = pl.concat([
    subset.select("n_partitions", "worker", pl.col("peak_rss_mb").alias("value")),
    subset.filter(pl.col("worker") == "pure_polars").select(
        "n_partitions",
        pl.lit("dataset_size").alias("worker"),
        pl.col("dataset_size_mb").alias("value"),
    ),
])

chart = (
    alt
    .Chart(long_df)
    .mark_line()
    .encode(
        x=alt.X("n_partitions:Q", title="n_partitions"),
        y=alt.Y("value:Q", title="MB"),
        color=alt.Color(
            "worker:N",
            scale=alt.Scale(
                domain=["pure_polars", "by_partition", "dataset_size"],
                range=["#0075ff", "#F28E2B", "#000000"],
            ),
            legend=alt.Legend(title=None),
        ),
        strokeDash=alt.StrokeDash(
            "worker:N",
            scale=alt.Scale(
                domain=["pure_polars", "by_partition", "dataset_size"],
                range=[[1, 0], [1, 0], [6, 4]],
            ),
            legend=None,
        ),
    )
    .properties(width=600, height=350)
)

chart.save("dwp-vs-polars-partitioned.png", scale_factor=4)
display(chart)  # type: ignore

alt.Chart(...)